### ЗАДАЧА: Панель претензий к поставщикам по недопоставке по паттерну `MVC`

Команда procurement operations разбирает кейсы по недопоставке товара от поставщиков.
После приемки поставки система сравнивает ожидаемое количество и фактически полученный товар.
Если есть расхождение, создается кейс, по которому нужно провести расследование, запросить документы,
согласовать компенсацию от поставщика и закрыть кейс.

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за вывод данных;
- `Controller` принимает действия и связывает `Model` и `View`.

## Что должно храниться в кейсе

Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `supplier` — поставщик;
- `shipment_id` — идентификатор поставки;
- `sku` — товар;
- `expected_qty` — ожидаемое количество;
- `received_qty` — фактически принятое количество;
- `unit_cost` — себестоимость единицы товара;
- `claim_qty` — количество товара, которое считается недопоставленным;
- `claim_amount` — сумма претензии к поставщику;
- `approved_compensation` — согласованная компенсация;
- `remaining_loss` — остаток убытка после компенсации;
- `status` — текущий статус кейса;
- `manager` — сотрудник, который ведет кейс;
- `documents_verified` — подтверждены ли документы;
- `decision` — итоговое решение.

## Формулы

При создании кейса и после любого изменения компенсации нужно правильно считать:
- `claim_qty = expected_qty - received_qty`
- если `claim_qty < 0`, нужно выбрасывать ошибку;
- `claim_amount = claim_qty * unit_cost`
- `remaining_loss = claim_amount - approved_compensation`
- все денежные значения нужно округлять до 2 знаков.

## Статусы кейса

- `new`
- `investigating`
- `documents_requested`
- `documents_verified`
- `ready_for_resolution`
- `fully_compensated`
- `partial_compensated`
- `rejected`
- `escalated`

## Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `manager` несуществующему кейсу;
- финальные кейсы (`fully_compensated`, `partial_compensated`, `rejected`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `manager`;
- запросить документы можно только из `investigating`;
- подтвердить документы можно только из `documents_requested`;
- при подтверждении документов поле `documents_verified` должно стать `True`, а статус — `documents_verified`;
- установить `approved_compensation` можно только из `investigating` или `documents_verified`;
- `approved_compensation` не может быть меньше `0`;
- `approved_compensation` не может быть больше `claim_amount`;
- после изменения `approved_compensation` нужно пересчитать `remaining_loss`;
- перевод в `ready_for_resolution` возможен только из `investigating` или `documents_verified`;
- перевод в `ready_for_resolution` невозможен, если `approved_compensation == 0` и при этом `documents_verified == False`;
- завершить кейс как `fully_compensated` можно только из `ready_for_resolution`, если `approved_compensation == claim_amount`;
- завершить кейс как `partial_compensated` можно только из `ready_for_resolution`, если `0 < approved_compensation < claim_amount`;
- завершить кейс как `rejected` можно только из `ready_for_resolution`, если `approved_compensation == 0`;
- эскалировать кейс можно только из `investigating`, `documents_verified` или `ready_for_resolution`;
- при любом финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать менеджера;
- начинать расследование;
- запрашивать документы;
- подтверждать документы;
- устанавливать `approved_compensation`;
- переводить кейс в `ready_for_resolution`;
- завершать кейс как `fully_compensated`;
- завершать кейс как `partial_compensated`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов;
- возвращать summary.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

Если список кейсов пустой, вывести отдельное сообщение.

## Что должен делать `Controller`

Контроллер должен:
- вызывать методы модели;
- оборачивать операции в `try/except` с `ValueError`;
- передавать результат во view;
- обработать все действия из `actions`.

## Формат строки кейса

Каждый кейс можно вывести строкой такого вида:

`case_id | supplier | shipment_id | sku | expected_qty | received_qty | claim_qty | claim_amount | approved_compensation | remaining_loss | status | manager | documents_verified | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_claim_amount` — общая сумма претензий;
- `total_approved_compensation` — общая согласованная компенсация;
- `total_remaining_loss` — общий остаток убытка;
- `verified_docs_cases` — количество кейсов, где документы подтверждены;
- `fully_compensated_amount` — сумма компенсаций по кейсам со статусом `fully_compensated`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить данные из `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести финальное состояние кейсов и summary.

In [5]:
initial_cases = [
    ("SC-100", "alpha-supply", "SHIP-7001", "SKU-100", 120, 102, 35.0),
    ("SC-101", "beta-distribution", "SHIP-7002", "SKU-200", 80, 80, 50.0),
]

actions = [
    ("show",),
    ("investigate", "SC-100"),
    ("assign", "SC-100", "Olga"),
    ("investigate", "SC-100"),
    ("docs_request", "SC-100"),
    ("docs_verify", "SC-100"),
    ("set_compensation", "SC-100", 500.0),
    ("ready", "SC-100"),
    ("partial", "SC-100", "supplier_accepted_partial_claim"),
    ("create", "SC-102", "gamma-warehouses", "SHIP-7003", "SKU-300", 60, 40, 28.0),
    ("assign", "SC-102", "Max"),
    ("investigate", "SC-102"),
    ("set_compensation", "SC-102", 560.0),
    ("ready", "SC-102"),
    ("full", "SC-102", "full_compensation_approved"),
    ("create", "SC-103", "delta-trade", "SHIP-7004", "SKU-400", 45, 30, 42.0),
    ("assign", "SC-103", "Ina"),
    ("investigate", "SC-103"),
    ("escalate", "SC-103", "supplier_disputes_shortage"),
    ("show",),
]

class Case:
    def __init__(self, case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost):
        if received_qty > expected_qty:
            raise ValueError("Received quantity cannot be greater than expected.")
        self.case_id = case_id
        self.supplier = supplier
        self.shipment_id = shipment_id
        self.sku = sku
        self.expected_qty = expected_qty
        self.received_qty = received_qty
        self.unit_cost = unit_cost
        self.claim_qty = self.expected_qty - self.received_qty
        if self.claim_qty < 0:
            raise ValueError("Claim quantity cannot be negative.")
        self.claim_amount = round(self.claim_qty * self.unit_cost, 2)
        self.approved_compensation = 0.0
        self.remaining_loss = self.claim_amount - self.approved_compensation
        self.status = 'new'
        self.manager = None
        self.documents_verified = False
        self.decision = None

class Model:
    def __init__(self):
        self.cases = {}

    def create_case(self, case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost):
        if case_id in self.cases:
            raise ValueError(f"Case with id {case_id} already exists.")
        case = Case(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost)
        self.cases[case_id] = case

    def get_case(self, case_id):
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found.")
        return self.cases[case_id]

    def assign_manager(self, case_id, manager_name):
        case = self.get_case(case_id)
        if case.status in ['fully_compensated', 'partial_compensated', 'rejected', 'escalated']:
            raise ValueError("Cannot assign manager to a final case.")
        case.manager = manager_name

    def investigate(self, case_id):
        case = self.get_case(case_id)
        if case.status != 'new':
            raise ValueError("Can only investigate from 'new' status.")
        if not case.manager:
            raise ValueError("Manager must be assigned before investigation.")
        case.status = 'investigating'

    def request_documents(self, case_id):
        case = self.get_case(case_id)
        if case.status != 'investigating':
            raise ValueError("Documents can only be requested from 'investigating' status.")
        case.status = 'documents_requested'

    def verify_documents(self, case_id):
        case = self.get_case(case_id)
        if case.status != 'documents_requested':
            raise ValueError("Documents can only be verified from 'documents_requested' status.")
        case.documents_verified = True
        case.status = 'documents_verified'

    def set_approved_compensation(self, case_id, amount):
        case = self.get_case(case_id)
        if case.status not in ['investigating', 'documents_verified']:
            raise ValueError("Can only set compensation during investigation or after documents verified.")
        if amount < 0:
            raise ValueError("Approved compensation cannot be negative.")
        if amount > case.claim_amount:
            raise ValueError("Approved compensation cannot exceed claim amount.")
        case.approved_compensation = round(amount, 2)
        case.remaining_loss = round(case.claim_amount - case.approved_compensation, 2)

    def move_to_ready(self, case_id):
        case = self.get_case(case_id)
        if case.status not in ['investigating', 'documents_verified']:
            raise ValueError("Can only move to ready from investigation or documents verified.")
        if case.approved_compensation == 0 and not case.documents_verified:
            raise ValueError("Cannot move to ready if compensation is 0 and documents are not verified.")
        if case.status == 'ready_for_resolution':
            return
        if case.status == 'fully_compensated' or case.status == 'partial_compensated' or case.status == 'rejected':
            raise ValueError("Cannot move to ready from a final status.")
        case.status = 'ready_for_resolution'

    def close_fully_compensated(self, case_id):
        case = self.get_case(case_id)
        if case.status != 'ready_for_resolution':
            raise ValueError("Can only close fully compensated from 'ready_for_resolution'.")
        if case.approved_compensation != case.claim_amount:
            raise ValueError("Approved compensation must be equal to claim amount.")
        case.status = 'fully_compensated'
        case.decision = 'Fully compensated'

    def close_partial_compensated(self, case_id):
        case = self.get_case(case_id)
        if case.status != 'ready_for_resolution':
            raise ValueError("Can only close partial compensation from 'ready_for_resolution'.")
        if 0 < case.approved_compensation < case.claim_amount:
            case.status = 'partial_compensated'
            case.decision = 'Partial compensation'
        else:
            raise ValueError("Approved amount must be between 0 and claim amount.")

    def reject_case(self, case_id):
        case = self.get_case(case_id)
        if case.status != 'ready_for_resolution':
            raise ValueError("Can only reject from 'ready_for_resolution'.")
        if case.approved_compensation == 0:
            case.status = 'rejected'
            case.decision = 'Rejected'

    def escalate(self, case_id, reason):
        case = self.get_case(case_id)
        if case.status not in ['investigating', 'documents_verified', 'ready_for_resolution']:
            raise ValueError("Can only escalate from investigating, documents verified, or ready_for_resolution.")
        case.status = 'escalated'
        case.decision = reason

    def get_cases(self):
        return list(self.cases.values())

    def get_summary(self):
        stats = {
            'new': 0,
            'investigating': 0,
            'documents_requested': 0,
            'documents_verified': 0,
            'ready_for_resolution': 0,
            'fully_compensated': 0,
            'partial_compensated': 0,
            'rejected': 0,
            'escalated': 0,
            'total_claim_amount': 0.0,
            'total_approved_compensation': 0.0,
            'total_remaining_loss': 0.0,
            'verified_docs_cases': 0,
            'fully_compensated_amount': 0.0,
        }
        for case in self.cases.values():
            stats[case.status] += 1
            stats['total_claim_amount'] += case.claim_amount
            stats['total_approved_compensation'] += case.approved_compensation
            stats['total_remaining_loss'] += case.remaining_loss
            if case.documents_verified:
                stats['verified_docs_cases'] += 1
            if case.status == 'fully_compensated':
                stats['fully_compensated_amount'] += case.approved_compensation
        for key in ['total_claim_amount', 'total_approved_compensation', 'total_remaining_loss', 'fully_compensated_amount']:
            stats[key] = round(stats[key], 2)
        return stats
    
class View:
    def show_cases(self, cases):
        if not cases:
            print("There are no cases to display.")
            return
        header = " | ".join([
            "case_id", "supplier", "shipment_id", "sku", "expected_qty", "received_qty",
            "claim_qty", "claim_amount", "approved_compensation", "remaining_loss",
            "status", "manager", "docs_verified", "decision"
        ])
        print(header)
        for case in cases:
            line = " | ".join([
                case.case_id, case.supplier, case.shipment_id, case.sku,
                str(case.expected_qty), str(case.received_qty),
                str(case.claim_qty), f"{case.claim_amount:.2f}",
                f"{case.approved_compensation:.2f}", f"{case.remaining_loss:.2f}",
                case.status, case.manager or "-", str(case.documents_verified),
                case.decision or "-"
            ])
            print(line)

    def show_summary(self, summary):
        print("Case statistics:")
        for key, value in summary.items():
            print(f"{key}: {value}")

    def show_message(self, message):
        print(f" {message}")

    def show_error(self, error):
        print(f"Error: {error}")

class Controller:
    def __init__(self, model, view):
        self.model = model
        self.view = view

    def create_case(self, case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost):
        try:
            self.model.create_case(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost)
            self.view.show_message(f"Case {case_id} successfully created.")
        except ValueError as e:
            self.view.show_error(str(e))

    def assign(self, case_id, manager):
        try:
            self.model.assign_manager(case_id, manager)
            self.view.show_message(f"Manager appointed {manager} for the case {case_id}.")
        except ValueError as e:
            self.view.show_error(str(e))

    def investigate(self, case_id):
        try:
            self.model.investigate(case_id)
            self.view.show_message(f"An investigation into the case has been launched {case_id}.")
        except ValueError as e:
            self.view.show_error(str(e))

    def request_documents(self, case_id):
        try:
            self.model.request_documents(case_id)
            self.view.show_message(f"Documents on the case have been requested {case_id}.")
        except ValueError as e:
            self.view.show_error(str(e))

    def verify_documents(self, case_id):
        try:
            self.model.verify_documents(case_id)
            self.view.show_message(f"Documents confirmed for the case {case_id}.")
        except ValueError as e:
            self.view.show_error(str(e))

    def set_compensation(self, case_id, amount):
        try:
            self.model.set_approved_compensation(case_id, amount)
            self.view.show_message(f"Compensation has been established {amount} for the case {case_id}.")
        except ValueError as e:
            self.view.show_error(str(e))

    def move_to_ready(self, case_id):
        try:
            self.model.move_to_ready(case_id)
            self.view.show_message(f"Case {case_id} transferred to state 'ready_for_resolution'.")
        except ValueError as e:
            self.view.show_error(str(e))

    def close_fully(self, case_id):
        try:
            self.model.close_fully_compensated(case_id)
            self.view.show_message(f"Case {case_id} closed as 'fully_compensated'.")
        except ValueError as e:
            self.view.show_error(str(e))

    def close_partial(self, case_id):
        try:
            self.model.close_partial_compensated(case_id)
            self.view.show_message(f"Case {case_id} closed as 'partial_compensated'.")
        except ValueError as e:
            self.view.show_error

    def close_partial(self, case_id):
        try:
            self.model.close_partial_compensated(case_id)
            self.view.show_message(f"Case {case_id} closed as 'partial_compensated'.")
        except ValueError as e:
            self.view.show_error(str(e))

    def reject(self, case_id):
        try:
            self.model.reject_case(case_id)
            self.view.show_message(f"Case {case_id} rejected.")
        except ValueError as e:
            self.view.show_error(str(e))

    def escalate(self, case_id, reason):
        try:
            self.model.escalate(case_id, reason)
            self.view.show_message(f"Case {case_id} escalated: {reason}.")
        except ValueError as e:
            self.view.show_error(str(e))

    def show_cases(self):
        cases = self.model.get_cases()
        self.view.show_cases(cases)

    def show_summary(self):
        summary = self.model.get_summary()
        self.view.show_summary(summary)        

if __name__ == "__main__":
    model = Model()
    view = View()
    ctrl = Controller(model, view)

    ctrl.create_case("C1", "Supplier A", "SHP101", "SKU1", 10, 8, 500.0)
    ctrl.create_case("C2", "Supplier B", "SHP102", "SKU2", 20, 20, 200.0)  
    ctrl.create_case("C3", "Supplier C", "SHP103", "SKU3", 100, 80, 10.0)

    ctrl.assign("C1", "Ivanov")
    ctrl.investigate("C1")

    ctrl.request_documents("C1")
    ctrl.verify_documents("C1")

    ctrl.set_compensation("C1", 1000.0) 
    ctrl.move_to_ready("C1")
    ctrl.close_partial("C1") 

    ctrl.assign("C3", "Petrov")
    ctrl.investigate("C3")
    ctrl.set_compensation("C3", 200.0)
    ctrl.move_to_ready("C3")
    ctrl.close_fully("C3")

    ctrl.assign("C2", "Ivanov")
    ctrl.investigate("C2")
    ctrl.set_compensation("C2", 0.0)
    ctrl.move_to_ready("C2")
    ctrl.reject("C2")

    print("\nAll cases")
    ctrl.show_cases()

    print("\nSummary")
    ctrl.show_summary()

 Case C1 successfully created.
 Case C2 successfully created.
 Case C3 successfully created.
 Manager appointed Ivanov for the case C1.
 An investigation into the case has been launched C1.
 Documents on the case have been requested C1.
 Documents confirmed for the case C1.
 Compensation has been established 1000.0 for the case C1.
 Case C1 transferred to state 'ready_for_resolution'.
Error: Approved amount must be between 0 and claim amount.
 Manager appointed Petrov for the case C3.
 An investigation into the case has been launched C3.
 Compensation has been established 200.0 for the case C3.
 Case C3 transferred to state 'ready_for_resolution'.
 Case C3 closed as 'fully_compensated'.
 Manager appointed Ivanov for the case C2.
 An investigation into the case has been launched C2.
 Compensation has been established 0.0 for the case C2.
Error: Cannot move to ready if compensation is 0 and documents are not verified.
Error: Can only reject from 'ready_for_resolution'.

All cases
case_id